# Continual Learning with MNIST

This notebook prepares the MNIST dataset for the mini-project **Continual Learning Neural Networks for Lifelong Image Classification without Catastrophic Forgetting**.

We split MNIST into five sequential tasks:
- Task 1: digits 0 and 1
- Task 2: digits 2 and 3
- Task 3: digits 4 and 5
- Task 4: digits 6 and 7
- Task 5: digits 8 and 9

The same task split can later be used for Naive Sequential Training, EWC and Experience Replay.

In [ ]:
import gzip
import struct
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

print('Libraries imported successfully.')

## 1. Load the raw MNIST files

The four compressed MNIST files are stored in the repository root.

In [ ]:
DATA_DIR = Path('.')

TRAIN_IMAGES = DATA_DIR / 'train-images-idx3-ubyte.gz'
TRAIN_LABELS = DATA_DIR / 'train-labels-idx1-ubyte.gz'
TEST_IMAGES = DATA_DIR / 't10k-images-idx3-ubyte.gz'
TEST_LABELS = DATA_DIR / 't10k-labels-idx1-ubyte.gz'

for file_path in [TRAIN_IMAGES, TRAIN_LABELS, TEST_IMAGES, TEST_LABELS]:
    print(file_path, '->', 'FOUND' if file_path.exists() else 'MISSING')

In [ ]:
def load_idx_images(path):
    with gzip.open(path, 'rb') as f:
        magic, n_images, rows, cols = struct.unpack('>IIII', f.read(16))
        if magic != 2051:
            raise ValueError(f'Invalid image file magic number: {magic}')
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(n_images, rows, cols)


def load_idx_labels(path):
    with gzip.open(path, 'rb') as f:
        magic, n_labels = struct.unpack('>II', f.read(8))
        if magic != 2049:
            raise ValueError(f'Invalid label file magic number: {magic}')
        return np.frombuffer(f.read(), dtype=np.uint8)


x_train = load_idx_images(TRAIN_IMAGES)
y_train = load_idx_labels(TRAIN_LABELS)
x_test = load_idx_images(TEST_IMAGES)
y_test = load_idx_labels(TEST_LABELS)

print('Training images:', x_train.shape)
print('Training labels:', y_train.shape)
print('Test images:', x_test.shape)
print('Test labels:', y_test.shape)

## 2. Preprocess the images

MNIST pixel values are between 0 and 255. We scale them to values between 0 and 1.

In [ ]:
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

print('Training pixel range:', x_train.min(), 'to', x_train.max())
print('Test pixel range:', x_test.min(), 'to', x_test.max())

## 3. Look at a few MNIST images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit in range(10):
    index = np.where(y_train == digit)[0][0]
    ax = axes.flat[digit]
    ax.imshow(x_train[index], cmap='gray')
    ax.set_title(f'Digit {digit}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Split MNIST into five continual-learning tasks

Each task contains two digit classes. The model will learn these tasks one after another.

In [ ]:
TASK_DIGITS = {
    1: (0, 1),
    2: (2, 3),
    3: (4, 5),
    4: (6, 7),
    5: (8, 9),
}


def create_task(images, labels, digits):
    mask = np.isin(labels, digits)
    return images[mask], labels[mask]


train_tasks = {}
test_tasks = {}

for task_id, digits in TASK_DIGITS.items():
    train_tasks[task_id] = create_task(x_train, y_train, digits)
    test_tasks[task_id] = create_task(x_test, y_test, digits)

    x_task, y_task = train_tasks[task_id]
    print(f'Task {task_id}: digits {digits} -> {len(x_task)} training images')

## 5. Verify the task split

In [ ]:
for task_id in train_tasks:
    _, train_labels = train_tasks[task_id]
    _, test_labels = test_tasks[task_id]
    print(
        f'Task {task_id}: '
        f'train classes = {np.unique(train_labels)}, '
        f'test classes = {np.unique(test_labels)}'
    )

## 6. Build the Naive Sequential Neural Network baseline

This is our **baseline**. It does not use EWC or Experience Replay.

The same neural network is trained on Task 1, then Task 2, then Task 3, Task 4 and Task 5. **We do not reset the model between tasks.**

We use one output layer with 10 classes (digits 0 to 9). This means the model must keep recognising old digits while new digit classes are introduced.

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### 6.1 Create PyTorch DataLoaders

A DataLoader sends the images to the neural network in small batches instead of loading the whole task into the model at once.

In [ ]:
BATCH_SIZE = 128

def make_loader(task_data, batch_size=BATCH_SIZE, shuffle=False):
    images, labels = task_data
    x = torch.tensor(images, dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(labels, dtype=torch.long)
    dataset = TensorDataset(x, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loaders = {
    task_id: make_loader(train_tasks[task_id], shuffle=True)
    for task_id in TASK_DIGITS
}

test_loaders = {
    task_id: make_loader(test_tasks[task_id], shuffle=False)
    for task_id in TASK_DIGITS
}

print('DataLoaders created for all five tasks.')

### 6.2 Define the neural network

MNIST images are 28 × 28 pixels. We flatten each image into 784 numbers, pass them through two hidden layers, and use 10 output neurons for digits 0 to 9.

We will use this **same architecture later for EWC and Experience Replay** so that the comparison is fair.

In [ ]:
class MNISTNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model = MNISTNetwork().to(device)
print(model)

### 6.3 Training and evaluation functions

`train_one_task` trains the model on one task. `evaluate` measures classification accuracy on a task's test set.

In [ ]:
def train_one_task(model, loader, optimizer, criterion, epochs):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total
        print(
            f'  Epoch {epoch + 1}/{epochs} | '
            f'Loss: {epoch_loss:.4f} | '
            f'Train accuracy: {epoch_accuracy * 100:.2f}%'
        )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return correct / total

### 6.4 Train sequentially from Task 1 to Task 5

This is the important continual-learning step. The model is trained on one task at a time in this order:

**(0,1) → (2,3) → (4,5) → (6,7) → (8,9)**

After learning each new task, we test the model on **every task it has seen so far**. If accuracy on an older task drops, that is evidence of catastrophic forgetting.

In [ ]:
LEARNING_RATE = 0.001
EPOCHS_PER_TASK = 5

# Start with a fresh model for the naive baseline.
torch.manual_seed(SEED)
naive_model = MNISTNetwork().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(naive_model.parameters(), lr=LEARNING_RATE)

# Row = task just learned, Column = task being tested.
accuracy_matrix = np.full((5, 5), np.nan, dtype=np.float64)

for current_task in range(1, 6):
    print('\n' + '=' * 65)
    print(f'TRAINING TASK {current_task}: digits {TASK_DIGITS[current_task]}')
    print('=' * 65)

    train_one_task(
        naive_model,
        train_loaders[current_task],
        optimizer,
        criterion,
        EPOCHS_PER_TASK
    )

    print('\nTesting all tasks learned so far:')
    for test_task in range(1, current_task + 1):
        accuracy = evaluate(naive_model, test_loaders[test_task])
        accuracy_matrix[current_task - 1, test_task - 1] = accuracy
        print(
            f'  Task {test_task} {TASK_DIGITS[test_task]}: '
            f'{accuracy * 100:.2f}%'
        )

## 7. Inspect the accuracy matrix

Each row shows the model **after learning a task**. Each column shows the accuracy on a particular task.

For example, the Task 1 column shows whether the model still remembers digits 0 and 1 after learning Tasks 2, 3, 4 and 5.

In [ ]:
column_names = [f'Task {i}' for i in range(1, 6)]

print('Naive Sequential Accuracy Matrix (%)\n')
print(' ' * 14 + ''.join(f'{name:>12}' for name in column_names))

for row in range(5):
    values = []
    for col in range(5):
        value = accuracy_matrix[row, col]
        values.append('      -     ' if np.isnan(value) else f'{value * 100:11.2f}%')
    print(f'After Task {row + 1:<2} ' + ''.join(values))

## 8. Calculate final average accuracy and forgetting

**Final Average Accuracy** tells us how well the model performs across all five tasks after it has finished learning Task 5.

**Forgetting** measures how much performance an old task lost between its best recorded accuracy and its final accuracy.

In [ ]:
final_accuracies = accuracy_matrix[-1, :]
final_average_accuracy = np.nanmean(final_accuracies)

forgetting_per_task = []

# We calculate forgetting for Tasks 1 to 4.
# Task 5 is the final task, so there is no later task that can make it forget yet.
for task_index in range(4):
    history = accuracy_matrix[task_index:, task_index]
    history = history[~np.isnan(history)]
    best_accuracy = np.max(history)
    final_accuracy = accuracy_matrix[-1, task_index]
    forgetting = best_accuracy - final_accuracy
    forgetting_per_task.append(forgetting)

average_forgetting = np.mean(forgetting_per_task)

print(f'Final Average Accuracy: {final_average_accuracy * 100:.2f}%')
print(f'Average Forgetting:      {average_forgetting * 100:.2f} percentage points')

for i, forgetting in enumerate(forgetting_per_task, start=1):
    print(f'Task {i} forgetting: {forgetting * 100:.2f} percentage points')

## 9. Visualise catastrophic forgetting

The graph below shows how the accuracy of each task changes as the model learns more tasks. A strong downward trend for an old task means the model is forgetting it.

In [ ]:
plt.figure(figsize=(10, 6))

for task_index in range(5):
    x_values = []
    y_values = []

    for after_task in range(5):
        value = accuracy_matrix[after_task, task_index]
        if not np.isnan(value):
            x_values.append(after_task + 1)
            y_values.append(value * 100)

    plt.plot(
        x_values,
        y_values,
        marker='o',
        label=f'Task {task_index + 1} {TASK_DIGITS[task_index + 1]}'
    )

plt.xlabel('Task most recently learned')
plt.ylabel('Test accuracy (%)')
plt.title('Naive Sequential Training: Accuracy Across Tasks')
plt.xticks(range(1, 6))
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 10. Save the naive baseline results

These files make it easier to compare the naive baseline with EWC and Experience Replay later.

In [ ]:
np.savetxt(
    'naive_accuracy_matrix.csv',
    accuracy_matrix,
    delimiter=',',
    fmt='%.6f',
    header='Task1,Task2,Task3,Task4,Task5',
    comments=''
)

torch.save(naive_model.state_dict(), 'naive_sequential_mnist.pt')

print('Saved: naive_accuracy_matrix.csv')
print('Saved: naive_sequential_mnist.pt')

## What this baseline gives us

The naive model gives us the **problem case** for the project. It learns each new MNIST task without any mechanism for protecting old knowledge.

The next continual-learning models should use the **same dataset split, neural-network architecture, batch size, learning rate and epochs**. We can then compare Naive Sequential Training against **EWC** and **Experience Replay** using final accuracy and forgetting.

## 11. Elastic Weight Consolidation (EWC)

Now we build the first continual-learning method.

**EWC tries to stop catastrophic forgetting by protecting weights that were important for previous tasks.** After each task, we estimate the importance of every trainable parameter using the Fisher Information. When the model learns the next task, changing an important old parameter creates an extra penalty in the loss.

The EWC training loss is:

$$L_{total}=L_{task}+\frac{\lambda}{2}\sum_i F_i(\theta_i-\theta_i^*)^2$$

where $F_i$ is the importance of parameter $i$, $\theta_i^*$ is its saved value after an old task, and $\lambda$ controls how strongly old knowledge is protected.

### 11.1 Estimate parameter importance with the empirical Fisher Information

In simple terms, we pass examples from the task through the trained model, calculate gradients, square those gradients, and average them. A larger value means that parameter was more important for the task.

In [ ]:
def calculate_fisher(model, loader, criterion, max_batches=50):
    """Estimate the diagonal empirical Fisher Information for one task."""
    model.eval()

    fisher = {
        name: torch.zeros_like(param, device=device)
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    batches_used = 0

    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break

        images = images.to(device)
        labels = labels.to(device)

        model.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher[name] += param.grad.detach() ** 2

        batches_used += 1

    if batches_used == 0:
        raise ValueError('No batches were available to estimate the Fisher Information.')

    for name in fisher:
        fisher[name] /= batches_used

    return fisher

### 11.2 Save the old weights and their importance

After a task is learned, EWC stores two things for that task:

1. a copy of the model parameters after the task, and
2. the Fisher importance values for those parameters.

With only five MNIST tasks, keeping one small EWC record for each previous task is manageable and keeps the implementation easy to understand.

In [ ]:
def save_ewc_memory(model, fisher):
    saved_parameters = {
        name: param.detach().clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    saved_fisher = {
        name: value.detach().clone()
        for name, value in fisher.items()
    }

    return {
        'parameters': saved_parameters,
        'fisher': saved_fisher
    }

### 11.3 Calculate the EWC penalty

For every previous task, the penalty checks how far the current parameters have moved from their saved values. Moving a parameter with a high Fisher importance value costs more than moving an unimportant parameter.

In [ ]:
def calculate_ewc_penalty(model, ewc_memory):
    if len(ewc_memory) == 0:
        return torch.tensor(0.0, device=device)

    penalty = torch.tensor(0.0, device=device)

    for memory in ewc_memory:
        old_parameters = memory['parameters']
        fisher = memory['fisher']

        for name, param in model.named_parameters():
            if name in fisher:
                penalty += torch.sum(
                    fisher[name] * (param - old_parameters[name]) ** 2
                )

    return penalty

### 11.4 Train one task using EWC

Task 1 is trained normally because there is no older task yet. From Task 2 onward, the EWC penalty is added to the normal cross-entropy loss.

In [ ]:
def train_one_task_ewc(
    model,
    loader,
    optimizer,
    criterion,
    epochs,
    ewc_memory,
    ewc_lambda
):
    model.train()

    for epoch in range(epochs):
        running_task_loss = 0.0
        running_ewc_penalty = 0.0
        correct = 0
        total = 0

        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            task_loss = criterion(outputs, labels)
            ewc_penalty = calculate_ewc_penalty(model, ewc_memory)

            loss = task_loss + (ewc_lambda / 2.0) * ewc_penalty
            loss.backward()
            optimizer.step()

            running_task_loss += task_loss.item() * labels.size(0)
            running_ewc_penalty += ewc_penalty.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        avg_task_loss = running_task_loss / total
        avg_penalty = running_ewc_penalty / total
        train_accuracy = correct / total

        print(
            f'  Epoch {epoch + 1}/{epochs} | '
            f'Task loss: {avg_task_loss:.4f} | '
            f'EWC penalty: {avg_penalty:.6f} | '
            f'Train accuracy: {train_accuracy * 100:.2f}%'
        )

### 11.5 Train EWC sequentially on all five tasks

We now repeat exactly the same task order used by the naive model. The network architecture, optimizer, learning rate, batch size and epochs stay the same. The important change is the EWC penalty.

`EWC_LAMBDA` is a tunable hyperparameter. A larger value protects old parameters more strongly, while a smaller value allows them to change more freely.

In [ ]:
EWC_LAMBDA = 1000.0
FISHER_BATCHES = 50

# Start EWC from the same random initialization used for the naive baseline.
torch.manual_seed(SEED)
ewc_model = MNISTNetwork().to(device)

ewc_optimizer = torch.optim.Adam(
    ewc_model.parameters(),
    lr=LEARNING_RATE
)

ewc_accuracy_matrix = np.full((5, 5), np.nan, dtype=np.float64)
ewc_memory = []

for current_task in range(1, 6):
    print('\n' + '=' * 70)
    print(f'EWC - TRAINING TASK {current_task}: digits {TASK_DIGITS[current_task]}')
    print('=' * 70)

    train_one_task_ewc(
        ewc_model,
        train_loaders[current_task],
        ewc_optimizer,
        criterion,
        EPOCHS_PER_TASK,
        ewc_memory,
        EWC_LAMBDA
    )

    print('\nTesting all tasks learned so far:')
    for test_task in range(1, current_task + 1):
        accuracy = evaluate(ewc_model, test_loaders[test_task])
        ewc_accuracy_matrix[current_task - 1, test_task - 1] = accuracy
        print(
            f'  Task {test_task} {TASK_DIGITS[test_task]}: '
            f'{accuracy * 100:.2f}%'
        )

    # We only need to protect Tasks 1-4 while later tasks are still coming.
    if current_task < 5:
        print(f'\nEstimating Fisher Information for Task {current_task}...')
        fisher = calculate_fisher(
            ewc_model,
            train_loaders[current_task],
            criterion,
            max_batches=FISHER_BATCHES
        )
        ewc_memory.append(save_ewc_memory(ewc_model, fisher))
        print(f'EWC memory now contains {len(ewc_memory)} previous task(s).')

## 12. Inspect the EWC accuracy matrix

In [ ]:
print('EWC Accuracy Matrix (%)\n')
print(' ' * 14 + ''.join(f'{name:>12}' for name in column_names))

for row in range(5):
    values = []
    for col in range(5):
        value = ewc_accuracy_matrix[row, col]
        values.append('      -     ' if np.isnan(value) else f'{value * 100:11.2f}%')
    print(f'After Task {row + 1:<2} ' + ''.join(values))

## 13. Calculate EWC accuracy and forgetting

We use the same metrics as the naive baseline so that the comparison is fair.

In [ ]:
def continual_learning_metrics(matrix):
    final_average_accuracy = np.nanmean(matrix[-1, :])
    forgetting_values = []

    for task_index in range(matrix.shape[1] - 1):
        history = matrix[task_index:, task_index]
        history = history[~np.isnan(history)]
        best_accuracy = np.max(history)
        final_accuracy = matrix[-1, task_index]
        forgetting_values.append(best_accuracy - final_accuracy)

    average_forgetting = np.mean(forgetting_values)
    return final_average_accuracy, average_forgetting, forgetting_values


ewc_final_accuracy, ewc_average_forgetting, ewc_forgetting = \
    continual_learning_metrics(ewc_accuracy_matrix)

print(f'EWC Final Average Accuracy: {ewc_final_accuracy * 100:.2f}%')
print(f'EWC Average Forgetting:      {ewc_average_forgetting * 100:.2f} percentage points')

for task_id, forgetting in enumerate(ewc_forgetting, start=1):
    print(
        f'Task {task_id} forgetting: '
        f'{forgetting * 100:.2f} percentage points'
    )

## 14. Compare Naive Sequential Training with EWC

The goal is not to assume that EWC must win. We compare the actual measurements produced by this notebook.

In [ ]:
naive_final_accuracy, naive_average_forgetting, _ = \
    continual_learning_metrics(accuracy_matrix)

print('FINAL COMPARISON')
print('-' * 65)
print(
    f"{'Method':<20} {'Final Avg Accuracy':>20} {'Avg Forgetting':>20}"
)
print('-' * 65)
print(
    f"{'Naive Sequential':<20} "
    f"{naive_final_accuracy * 100:>19.2f}% "
    f"{naive_average_forgetting * 100:>18.2f} pp"
)
print(
    f"{'EWC':<20} "
    f"{ewc_final_accuracy * 100:>19.2f}% "
    f"{ewc_average_forgetting * 100:>18.2f} pp"
)

### 14.1 Visual comparison

In [ ]:
methods = ['Naive Sequential', 'EWC']
final_accuracies = [naive_final_accuracy * 100, ewc_final_accuracy * 100]
average_forgetting_values = [
    naive_average_forgetting * 100,
    ewc_average_forgetting * 100
]

plt.figure(figsize=(7, 5))
plt.bar(methods, final_accuracies)
plt.ylabel('Final average accuracy (%)')
plt.title('Final Average Accuracy: Naive vs EWC')
plt.ylim(0, 100)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(methods, average_forgetting_values)
plt.ylabel('Average forgetting (percentage points)')
plt.title('Average Forgetting: Naive vs EWC')
plt.show()

## 15. Save the EWC results

In [ ]:
np.savetxt(
    'ewc_accuracy_matrix.csv',
    ewc_accuracy_matrix,
    delimiter=',',
    fmt='%.6f',
    header='Task1,Task2,Task3,Task4,Task5',
    comments=''
)

torch.save(ewc_model.state_dict(), 'ewc_mnist.pt')

print('Saved: ewc_accuracy_matrix.csv')
print('Saved: ewc_mnist.pt')

## 16. Experience Replay

Experience Replay uses a different strategy from EWC.

**Instead of protecting old weights, Replay stores a small number of examples from previous tasks and mixes them with the new task during training.**

The model therefore keeps seeing a little bit of old information while learning new classes. This acts like revision and helps reduce catastrophic forgetting.

We keep the same single 10-class neural network, task order, learning rate, batch size and epochs used by the Naive and EWC models.

### 16.1 Build a fixed-size replay buffer

The replay buffer has a fixed memory limit. In this notebook we use **1,000 stored images in total**.

As new digit classes are learned, the buffer is rebalanced so that the stored examples are spread roughly equally across all classes seen so far. This keeps the memory usage fixed instead of allowing it to grow after every task.

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=1000, seed=42):
        self.max_size = max_size
        self.images = torch.empty((0, 28, 28), dtype=torch.float32)
        self.labels = torch.empty((0,), dtype=torch.long)
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.labels)

    def sample(self, batch_size):
        if len(self) == 0:
            return None, None

        sample_size = min(batch_size, len(self))
        indices = self.rng.choice(len(self), size=sample_size, replace=False)
        indices = torch.tensor(indices, dtype=torch.long)

        replay_images = self.images[indices].unsqueeze(1)
        replay_labels = self.labels[indices]
        return replay_images, replay_labels

    def add_task_examples(self, task_data, seen_classes):
        """Keep a roughly equal number of examples for every class seen so far."""
        images, labels = task_data
        seen_classes = sorted(seen_classes)
        examples_per_class = self.max_size // len(seen_classes)

        new_images = []
        new_labels = []

        # First keep a reduced number of old examples already in memory.
        for class_id in seen_classes:
            old_indices = torch.where(self.labels == class_id)[0]

            if len(old_indices) > 0:
                keep = min(examples_per_class, len(old_indices))
                selected = self.rng.choice(
                    old_indices.numpy(),
                    size=keep,
                    replace=False
                )
                selected = torch.tensor(selected, dtype=torch.long)
                new_images.append(self.images[selected])
                new_labels.append(self.labels[selected])

        # Then add examples from the current task's new classes.
        current_classes = np.unique(labels)
        for class_id in current_classes:
            class_indices = np.where(labels == class_id)[0]
            take = min(examples_per_class, len(class_indices))
            selected = self.rng.choice(class_indices, size=take, replace=False)

            class_images = torch.tensor(images[selected], dtype=torch.float32)
            class_labels = torch.tensor(labels[selected], dtype=torch.long)

            # Remove an older copy of this class if one was already added above.
            if new_labels:
                filtered_images = []
                filtered_labels = []
                for imgs, labs in zip(new_images, new_labels):
                    if len(labs) == 0 or int(labs[0].item()) != int(class_id):
                        filtered_images.append(imgs)
                        filtered_labels.append(labs)
                new_images = filtered_images
                new_labels = filtered_labels

            new_images.append(class_images)
            new_labels.append(class_labels)

        self.images = torch.cat(new_images, dim=0)
        self.labels = torch.cat(new_labels, dim=0)

        # Safety check: never exceed the requested buffer size.
        if len(self) > self.max_size:
            indices = self.rng.choice(len(self), size=self.max_size, replace=False)
            indices = torch.tensor(indices, dtype=torch.long)
            self.images = self.images[indices]
            self.labels = self.labels[indices]

    def class_counts(self):
        return {
            int(class_id): int((self.labels == class_id).sum().item())
            for class_id in torch.unique(self.labels)
        }

### 16.2 Train one task using Experience Replay

For Task 1 the replay buffer is empty, so training is normal. From Task 2 onward, each current-task batch is combined with a small batch sampled from memory.

The loss is still ordinary cross-entropy. The important difference is that the batch now contains **new examples + old replay examples**.

In [ ]:
def train_one_task_replay(
    model,
    loader,
    optimizer,
    criterion,
    epochs,
    replay_buffer,
    replay_batch_size
):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for current_images, current_labels in loader:
            current_images = current_images.to(device)
            current_labels = current_labels.to(device)

            replay_images, replay_labels = replay_buffer.sample(replay_batch_size)

            if replay_images is not None:
                replay_images = replay_images.to(device)
                replay_labels = replay_labels.to(device)

                images = torch.cat([current_images, replay_images], dim=0)
                labels = torch.cat([current_labels, replay_labels], dim=0)
            else:
                images = current_images
                labels = current_labels

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total

        print(
            f'  Epoch {epoch + 1}/{epochs} | '
            f'Loss: {epoch_loss:.4f} | '
            f'Combined-batch accuracy: {epoch_accuracy * 100:.2f}%'
        )

### 16.3 Train Replay sequentially on all five tasks

We use a replay memory of 1,000 examples and sample up to 64 replay examples for each current-task batch.

After each task is finished, examples from that task are added to the buffer. When the next task begins, those stored examples become the model's revision material.

In [ ]:
REPLAY_BUFFER_SIZE = 1000
REPLAY_BATCH_SIZE = 64

# Use the same initial weights as the other two methods.
torch.manual_seed(SEED)
replay_model = MNISTNetwork().to(device)

replay_optimizer = torch.optim.Adam(
    replay_model.parameters(),
    lr=LEARNING_RATE
)

replay_buffer = ReplayBuffer(
    max_size=REPLAY_BUFFER_SIZE,
    seed=SEED
)

replay_accuracy_matrix = np.full((5, 5), np.nan, dtype=np.float64)
seen_classes = set()

for current_task in range(1, 6):
    print('\n' + '=' * 70)
    print(f'REPLAY - TRAINING TASK {current_task}: digits {TASK_DIGITS[current_task]}')
    print('=' * 70)
    print(f'Replay buffer before training: {len(replay_buffer)} examples')

    train_one_task_replay(
        replay_model,
        train_loaders[current_task],
        replay_optimizer,
        criterion,
        EPOCHS_PER_TASK,
        replay_buffer,
        REPLAY_BATCH_SIZE
    )

    print('\nTesting all tasks learned so far:')
    for test_task in range(1, current_task + 1):
        accuracy = evaluate(replay_model, test_loaders[test_task])
        replay_accuracy_matrix[current_task - 1, test_task - 1] = accuracy
        print(
            f'  Task {test_task} {TASK_DIGITS[test_task]}: '
            f'{accuracy * 100:.2f}%'
        )

    # Add the newly learned classes to memory after training the task.
    seen_classes.update(TASK_DIGITS[current_task])
    replay_buffer.add_task_examples(
        train_tasks[current_task],
        seen_classes
    )

    print(f'\nReplay buffer after Task {current_task}: {len(replay_buffer)} examples')
    print('Examples per stored class:', replay_buffer.class_counts())

## 17. Inspect the Experience Replay accuracy matrix

In [ ]:
print('Experience Replay Accuracy Matrix (%)\n')
print(' ' * 14 + ''.join(f'{name:>12}' for name in column_names))

for row in range(5):
    values = []
    for col in range(5):
        value = replay_accuracy_matrix[row, col]
        values.append('      -     ' if np.isnan(value) else f'{value * 100:11.2f}%')
    print(f'After Task {row + 1:<2} ' + ''.join(values))

## 18. Calculate Replay accuracy and forgetting

In [ ]:
replay_final_accuracy, replay_average_forgetting, replay_forgetting = \
    continual_learning_metrics(replay_accuracy_matrix)

print(f'Replay Final Average Accuracy: {replay_final_accuracy * 100:.2f}%')
print(f'Replay Average Forgetting:      {replay_average_forgetting * 100:.2f} percentage points')

for task_id, forgetting in enumerate(replay_forgetting, start=1):
    print(
        f'Task {task_id} forgetting: '
        f'{forgetting * 100:.2f} percentage points'
    )

## 19. Final comparison: Naive vs EWC vs Experience Replay

All three methods use the same MNIST task order and the same base neural-network architecture. We now compare them using the two main continual-learning measurements in this experiment:

- **Final Average Accuracy:** higher means the final model performs better across all five tasks.
- **Average Forgetting:** lower means the model retained more of its earlier knowledge.

In [ ]:
print('FINAL CONTINUAL-LEARNING COMPARISON')
print('-' * 72)
print(
    f"{'Method':<22} {'Final Avg Accuracy':>22} {'Avg Forgetting':>22}"
)
print('-' * 72)

comparison_results = [
    ('Naive Sequential', naive_final_accuracy, naive_average_forgetting),
    ('EWC', ewc_final_accuracy, ewc_average_forgetting),
    ('Experience Replay', replay_final_accuracy, replay_average_forgetting),
]

for method, avg_accuracy, avg_forgetting in comparison_results:
    print(
        f'{method:<22} '
        f'{avg_accuracy * 100:>21.2f}% '
        f'{avg_forgetting * 100:>20.2f} pp'
    )

### 19.1 Final average accuracy graph

In [ ]:
methods = [result[0] for result in comparison_results]
final_accuracy_values = [result[1] * 100 for result in comparison_results]

plt.figure(figsize=(8, 5))
plt.bar(methods, final_accuracy_values)
plt.ylabel('Final average accuracy (%)')
plt.title('Final Average Accuracy Across Continual-Learning Methods')
plt.ylim(0, 100)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

### 19.2 Average forgetting graph

In [ ]:
forgetting_values = [result[2] * 100 for result in comparison_results]

plt.figure(figsize=(8, 5))
plt.bar(methods, forgetting_values)
plt.ylabel('Average forgetting (percentage points)')
plt.title('Average Forgetting Across Continual-Learning Methods')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

### 19.3 Compare the final accuracy on each task

In [ ]:
task_labels = [f'Task {i}' for i in range(1, 6)]

x_positions = np.arange(1, 6)

plt.figure(figsize=(10, 6))
plt.plot(
    x_positions,
    accuracy_matrix[-1, :] * 100,
    marker='o',
    label='Naive Sequential'
)
plt.plot(
    x_positions,
    ewc_accuracy_matrix[-1, :] * 100,
    marker='o',
    label='EWC'
)
plt.plot(
    x_positions,
    replay_accuracy_matrix[-1, :] * 100,
    marker='o',
    label='Experience Replay'
)

plt.xlabel('Task')
plt.ylabel('Final test accuracy (%)')
plt.title('Final Accuracy on Each MNIST Task')
plt.xticks(x_positions, task_labels)
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 20. Save Experience Replay and final comparison results

In [ ]:
np.savetxt(
    'replay_accuracy_matrix.csv',
    replay_accuracy_matrix,
    delimiter=',',
    fmt='%.6f',
    header='Task1,Task2,Task3,Task4,Task5',
    comments=''
)

torch.save(replay_model.state_dict(), 'experience_replay_mnist.pt')

with open('final_comparison.csv', 'w') as f:
    f.write('Method,Final_Average_Accuracy,Average_Forgetting\n')
    for method, avg_accuracy, avg_forgetting in comparison_results:
        f.write(f'{method},{avg_accuracy:.6f},{avg_forgetting:.6f}\n')

print('Saved: replay_accuracy_matrix.csv')
print('Saved: experience_replay_mnist.pt')
print('Saved: final_comparison.csv')

## Experiment complete

The notebook now contains the complete first experiment for the mini-project:

1. **Naive Sequential Training** — demonstrates catastrophic forgetting.
2. **Elastic Weight Consolidation (EWC)** — protects parameters that were important for previous tasks.
3. **Experience Replay** — stores a small set of old examples and reuses them while learning new tasks.

The measured results from this notebook can now be used for the Results, Discussion and Critical Evaluation sections of the project report.